In [10]:
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
import pandas as pd

Entire the desired output_directory, model_name, and metric we care about. Then run all the code below

In [52]:
output_dir = Path('/data/vision/polina/users/marcusbl/bin_class/outputs_mm_vs_ps')

model_name = 'model_auc'
metrics = ['auc', 'acc']

Parse results into Pandas DF

In [53]:
results = []

for test_dir in output_dir.iterdir():
    for run_dir in test_dir.iterdir():
        if not run_dir.is_dir():
            continue

        with open(run_dir / 'test_info' / 'metric_info.json') as f:
            metric_info = json.load(f)

        flat = {
            (model, metric): value
            for model, metrics in metric_info.items()
            for metric, value in metrics.items()
        }

        results.append({
            "test": test_dir.name,
            "run": run_dir.name,
            **flat
        })

df = pd.DataFrame(results).set_index(["test", "run"])

df.columns = pd.MultiIndex.from_tuples(df.columns)
df = df.sort_index(axis=1).sort_index()


Average/Variance for desired metrics over all runs for the desired model

In [54]:
m = df[model_name]

stats = m.groupby(level='test').agg(['mean', 'var'])

stats = stats.sort_values(by=('auc', 'mean'), ascending=False)

stats[[(metric, stat) for metric in metrics for stat in ['mean', 'var']]]

auc               acc          
               mean       var    mean       var
test                                           
test_stack2  0.8942  0.000572  0.8410  0.002083
test_mm02_2  0.8878  0.000704  0.8308  0.001899
test_mm00_2  0.8734  0.000253  0.8252  0.003115
test_stack3  0.8620  0.000220  0.7946  0.003010
test_mm00_1  0.8604  0.000461  0.7990  0.004273
quick_run    0.8590       NaN  0.8230       NaN
test_mm02_1  0.8560  0.000202  0.7932  0.002519
test_ps00_1  0.6990  0.001739  0.7468  0.005731
test_ps02_1  0.6672  0.006271  0.7468  0.005731